In [0]:
from pyspark.sql.functions import *
from delta.tables import *
from pyspark.sql.types import *

### Scenario:
Your `sales_fact` Delta table is loaded daily by an append-only pipeline. Today's incremental file, `sales_fact_incremental.csv`, arrived from the upstream source — but upstream jobs sometimes retry and re-send a batch that includes transactions already loaded yesterday. If you blindly append, you'll get duplicate transactions in the fact table. Your job is to make today's load **idempotent**.

**Problem:**

- Load `sales_fact_seed.csv` and write it as a managed Delta table named `sales_fact` (this represents "yesterday's" already-loaded state — do this once as setup).
- Read `sales_fact_incremental.csv` as today's incoming batch.
- Identify which incoming rows are genuinely new (i.e., their `transaction_id` does not already exist in `sales_fact`) using an anti-join against the existing table — do not use a merge for this one, use a left-anti join explicitly.
- Append only the genuinely new rows to `sales_fact`.
- Display the final `sales_fact` table, sorted by `transaction_id` ascending, and confirm no duplicate `transaction_id` exists.

**Schema (both files)**

| Column | Type |
| :--- | :--- |
| **transaction_id** | string |
| **customer_id** | string |
| **amount** | decimal(10,2) |
| **txn_date** | date |

**Expected Output — final `sales_fact` table**

| transaction_id | customer_id | amount | txn_date |
| :--- | :--- | :--- | :--- |
| T001 | C001 | 100.00 | 2024-05-01 |
| T002 | C002 | 250.00 | 2024-05-01 |
| T003 | C001 | 75.00 | 2024-05-02 |
| T004 | C003 | 300.00 | 2024-05-03 |
| T005 | C002 | 50.00 | 2024-05-03 |

In [0]:
schema = StructType(
    [
        StructField("transaction_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("amount", DecimalType(10,2)),
        StructField("txn_date", DateType())
    ]
)

## Loading seed data
df_sales_seed = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/sales_fact_seed.csv")
)

(
    df_sales_seed.write.format("delta")
    .mode("overwrite") ## since it is onetime seeding data
    .saveAsTable("pyspark_practice.default.sales_fact")
)

## Incremental loading of sales data
df_sales_inc = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/sales_fact_incremental.csv")
)

## Reading the delta table
df_sales_delta = DeltaTable.forName(spark, "pyspark_practice.default.sales_fact").toDF()

df_sales_new_records = (
    df_sales_inc.join(df_sales_delta, "transaction_id", "anti")
)

(
    df_sales_new_records.write.format("delta")
    .mode("append")
    .saveAsTable("pyspark_practice.default.sales_fact")
)

### Scenario:
You're loading `product_inventory_updates.csv` from a warehouse system into a Delta table. The upstream system occasionally sends bad rows — negative quantities, zero/negative prices — that would corrupt downstream inventory value calculations if loaded as-is. You need to validate every row against business rules before it's trusted, and gracefully handle the case where the source file itself might not exist.

**Problem:**

- Wrap your file read in a `try/except` block. If the file path is wrong or missing, catch the exception and print a clear error message instead of letting the job crash with a raw stack trace.
- Read the file with an explicit schema.
- Apply these business rules to each row:
  - `quantity` must be greater than or equal to 0.
  - `unit_price` must be greater than 0.
- Split the data into two sets:
  - **Valid rows** — pass both rules.
  - **Invalid rows** — fail one or both rules. Add a `reason` column listing which rule(s) were violated (e.g. `invalid_quantity`, `invalid_price`, or `invalid_quantity,invalid_price` if both fail).
- Write the invalid rows to a `product_inventory_quarantine` Delta table.
- From the valid rows only, compute `total_inventory_value` = sum of (`quantity` × `unit_price`).

**Schema**

| Column | Type |
| :--- | :--- |
| **product_id** | string |
| **product_name** | string |
| **quantity** | int |
| **unit_price** | decimal(10,2) |

**Expected Output — quarantine table (`product_inventory_quarantine`)**

| product_id | product_name | quantity | unit_price | reason |
| :--- | :--- | :--- | :--- | :--- |
| P002 | Widget B | -5 | 4.50 | invalid_quantity |
| P003 | Widget C | 20 | 0.00 | invalid_price |
| P006 | Widget F | -2 | -3.00 | invalid_quantity,invalid_price |

**Expected Output — total inventory value (valid rows only)**

| total_inventory_value |
| :--- |
| 949.50 |

In [0]:
path = "/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/product_inventory_updates.csv"
try:
    print("Reading product_inventory_updates.csv start")

    schema = StructType(
        [
            StructField("product_id", StringType()),
            StructField("product_name", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DecimalType(10,2))
        ]
    )

    product_inv_df = (
        spark.read.format("csv")
        .option("header", True)
        .schema(schema)
        .load(path)
    )
    print("Reading product_inventory_updates.csv successful")
except Exception as ex:
     print(f"Invalid path or File missing{ex}")
     raise

invalid_product_inv_df = (
    product_inv_df.filter(
        (col("quantity")<0) | (col("unit_price")<=0)
    )
    .withColumn(
        "reason",
        when(
            (col("quantity")<0) & (col("unit_price")<=0), 
            "invalid_quantity,invalid_price"
        )
        .when(col("unit_price")<=0, "invalid_price")
        .otherwise("invalid_quantity")
    )
)

invalid_product_inv_df.write.format("delta").mode("append").saveAsTable("pyspark_practice.default.product_inventory_quarantine")

valid_product_inv_df = (
    product_inv_df.filter(
        (col("quantity")>=0) & (col("unit_price")>0)
    )
    .withColumn(
        "inventory_value",
        col("quantity")*col("unit_price")
    )
)
display(valid_product_inv_df.agg(sum("inventory_value").alias("total_inventory_value")))